# Evidence-grounded biomedical question answering with hybrid RAG

CP5403 Assessment 2 - LLM project.

**Task:** answer PubMedQA research questions (yes / no / maybe) using an open-weights LLM that is grounded in retrieved PubMed abstracts.

**Pipeline:** BM25 + dense retrieval, fused with Reciprocal Rank Fusion, reranked by a cross-encoder, then fed to an open-weights generator.

**Experiments (ablation):** closed-book (no retrieval), BM25, dense, hybrid (RRF), hybrid + rerank, and an oracle upper bound that is given the correct abstract.

**Run order:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

*Research prototype only. Not medical advice.*

In [ ]:
!pip -q install rank_bm25 sentence-transformers datasets accelerate

In [ ]:
import re, time, json, random, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
N_EVAL = 200            # number of labelled questions used for evaluation
N_DISTRACTORS = 5000    # extra unlabelled abstracts that make retrieval harder
TOP_K = 3               # passages given to the generator
POOL = 50               # candidates kept per retriever
RERANK_POOL = 20        # candidates passed to the cross-encoder
MAX_WORDS_PER_PASSAGE = 300

GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
assert device == "cuda", "No GPU found. Use Runtime > Change runtime type > T4 GPU, then run again."

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("results", exist_ok=True)

## 1. Data

* `pqa_labeled`: 1,000 expert-labelled PubMedQA questions (the answer is yes / no / maybe).
* Each question comes with its abstract. The conclusion is removed from the abstract, so the answer is not leaked.
* The corpus is the 1,000 labelled abstracts plus 5,000 random abstracts from `pqa_unlabeled` as distractors.

In [ ]:
ds_l = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
ds_u = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")

docs, seen = [], set()
for r in ds_l:
    docs.append({"pubid": r["pubid"], "question": r["question"],
                 "text": " ".join(r["context"]["contexts"]), "label": r["final_decision"]})
    seen.add(r["pubid"])
n_labeled = len(docs)

rng = random.Random(SEED)
for i in rng.sample(range(len(ds_u)), N_DISTRACTORS + 200):
    r = ds_u[i]
    if len(docs) - n_labeled >= N_DISTRACTORS:
        break
    if r["pubid"] in seen:
        continue
    docs.append({"pubid": r["pubid"], "question": r["question"],
                 "text": " ".join(r["context"]["contexts"]), "label": None})
    seen.add(r["pubid"])

print("Corpus size:", len(docs), "| labelled:", n_labeled)
print(pd.Series([d["label"] for d in docs[:n_labeled]]).value_counts())

eval_idx = sorted(random.Random(SEED).sample(range(n_labeled), N_EVAL))
queries = [docs[i]["question"] for i in eval_idx]
gold_labels = [docs[i]["label"] for i in eval_idx]
majority = max(set(gold_labels), key=gold_labels.count)
majority_acc = gold_labels.count(majority) / len(gold_labels)
print(f"Majority-class baseline ('{majority}'): accuracy = {majority_acc:.3f}")

## 2. Retrievers

* **BM25** - keyword matching.
* **Dense** - `bge-small-en-v1.5` embeddings with cosine similarity.
* **Hybrid** - the BM25 and dense rankings merged with Reciprocal Rank Fusion (Cormack et al., 2009).
* **Hybrid + rerank** - the top 20 hybrid results rescored by a cross-encoder.

In [ ]:
def tokenize(s):
    return re.findall(r"[a-z0-9]+", s.lower())

corpus_texts = [d["text"] for d in docs]

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
embedder = SentenceTransformer(EMBED_MODEL, device=device)
doc_emb = embedder.encode(corpus_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
reranker = CrossEncoder(RERANK_MODEL, max_length=512, device=device)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def bm25_search(q, n=POOL):
    scores = bm25.get_scores(tokenize(q))
    return [int(i) for i in np.argsort(-scores)[:n]]

def dense_search(q, n=POOL):
    qe = embedder.encode(QUERY_PREFIX + q, normalize_embeddings=True)
    scores = doc_emb @ qe
    return [int(i) for i in np.argsort(-scores)[:n]]

def rrf_fuse(rank_lists, k=60):
    scores = {}
    for rl in rank_lists:
        for rank, d in enumerate(rl):
            scores[d] = scores.get(d, 0.0) + 1.0 / (k + rank + 1)
    return [d for d, _ in sorted(scores.items(), key=lambda x: -x[1])]

def hybrid_search(q, n=POOL):
    return rrf_fuse([bm25_search(q), dense_search(q)])[:n]

def hybrid_rerank_search(q, n=RERANK_POOL):
    cand = hybrid_search(q, RERANK_POOL)
    scores = reranker.predict([(q, corpus_texts[i]) for i in cand], batch_size=32, show_progress_bar=False)
    order = np.argsort(-scores)
    return [cand[int(j)] for j in order][:n]

## 3. Retrieval evaluation

Each question has one correct abstract (its own). We report Recall@1/3/5, Mean Reciprocal Rank (MRR) and average retrieval time per query.

In [ ]:
methods = {"bm25": bm25_search, "dense": dense_search,
           "hybrid_rrf": hybrid_search, "hybrid_rerank": hybrid_rerank_search}
gold_doc = eval_idx   # the correct abstract for question j is docs[eval_idx[j]]

rankings, ret_time = {}, {}
for name, fn in methods.items():
    ranks, times = [], []
    for q in queries:
        t = time.perf_counter()
        r = fn(q)
        times.append(time.perf_counter() - t)
        ranks.append(r)
    rankings[name] = ranks
    ret_time[name] = float(np.mean(times))

def retrieval_metrics(rank_lists, gold):
    out = {}
    for k in (1, 3, 5):
        out[f"recall@{k}"] = float(np.mean([g in r[:k] for r, g in zip(rank_lists, gold)]))
    rr = [1.0 / (r.index(g) + 1) if g in r else 0.0 for r, g in zip(rank_lists, gold)]
    out["mrr"] = float(np.mean(rr))
    return out

ret_rows = []
for name in methods:
    m = retrieval_metrics(rankings[name], gold_doc)
    m["retrieval_s"] = ret_time[name]
    ret_rows.append(pd.Series(m, name=name))
ret_df = pd.DataFrame(ret_rows).round(3)
ret_df.to_csv("results/retrieval_results.csv")
ret_df

## 4. Generator (open-weights LLM)

`Qwen2.5-3B-Instruct` runs in half precision on the free T4 GPU. Decoding is greedy, so results are repeatable. The model must answer with one word: yes, no or maybe.

In [ ]:
tok = AutoTokenizer.from_pretrained(GEN_MODEL)
llm = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.float16, device_map="auto")
llm.eval()

SYSTEM_RAG = ("You are a careful biomedical research assistant. Answer the question using ONLY the evidence "
              "provided. If the evidence is mixed or inconclusive, answer maybe.")
SYSTEM_CLOSED = ("You are a careful biomedical research assistant. Answer the question from your own knowledge. "
                 "If you are unsure or the evidence is mixed, answer maybe.")

def truncate(text, n=MAX_WORDS_PER_PASSAGE):
    w = text.split()
    return " ".join(w[:n])

def build_messages(question, passages):
    if passages:
        ev = "\n\n".join(f"[{i+1}] {truncate(corpus_texts[p])}" for i, p in enumerate(passages))
        user = f"Evidence:\n{ev}\n\nQuestion: {question}\n\nAnswer with exactly one word: yes, no, or maybe."
        system = SYSTEM_RAG
    else:
        user = f"Question: {question}\n\nAnswer with exactly one word: yes, no, or maybe."
        system = SYSTEM_CLOSED
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def parse_label(text):
    m = re.search(r"\b(yes|no|maybe)\b", text.lower())
    return (m.group(1), True) if m else ("maybe", False)

@torch.no_grad()
def generate_answer(question, passages):
    msgs = build_messages(question, passages)
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

## 5. Experiments

Six configurations are run on the same 200 questions.

In [ ]:
configs = {"closed_book": [None] * len(queries),
           "bm25": [rankings["bm25"][j][:TOP_K] for j in range(len(queries))],
           "dense": [rankings["dense"][j][:TOP_K] for j in range(len(queries))],
           "hybrid_rrf": [rankings["hybrid_rrf"][j][:TOP_K] for j in range(len(queries))],
           "hybrid_rerank": [rankings["hybrid_rerank"][j][:TOP_K] for j in range(len(queries))],
           "oracle": [[eval_idx[j]] for j in range(len(queries))]}

results, preds_store = {}, {}
for name, ctxs in configs.items():
    preds, valid, raws, times = [], [], [], []
    for q, p in zip(queries, ctxs):
        t = time.perf_counter()
        raw = generate_answer(q, p)
        times.append(time.perf_counter() - t)
        label, ok = parse_label(raw)
        preds.append(label); valid.append(ok); raws.append(raw)
    preds_store[name] = {"pred": preds, "raw": raws}
    results[name] = {"accuracy": accuracy_score(gold_labels, preds),
                     "macro_f1": f1_score(gold_labels, preds, average="macro", labels=["yes", "no", "maybe"], zero_division=0),
                     "invalid_rate": 1 - float(np.mean(valid)),
                     "generation_s": float(np.mean(times))}
    print(name, {k: round(v, 3) for k, v in results[name].items()})

## 6. Results table and charts

In [ ]:
res_df = pd.DataFrame(results).T
res_df = res_df.join(ret_df[["recall@1", "recall@3", "recall@5", "mrr", "retrieval_s"]])
res_df["total_s_per_question"] = res_df["generation_s"] + res_df["retrieval_s"].fillna(0)
res_df = res_df.round(3)
res_df.to_csv("results/results.csv")
print(f"Majority-class baseline accuracy: {majority_acc:.3f}")
res_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
res_df[["accuracy", "macro_f1"]].plot.bar(ax=ax[0], rot=30)
ax[0].axhline(majority_acc, color="grey", linestyle="--", label="majority baseline")
ax[0].set_title("Answer quality (200 questions)"); ax[0].set_ylim(0, 1); ax[0].legend()
ret_df[["recall@1", "recall@3", "recall@5", "mrr"]].plot.bar(ax=ax[1], rot=30)
ax[1].set_title("Retrieval quality"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.savefig("results/results.png", dpi=150); plt.show()

## 7. Error analysis

For the best RAG configuration: per-class scores, the confusion matrix, and accuracy when the correct abstract was or was not retrieved.

In [ ]:
rag_names = ["bm25", "dense", "hybrid_rrf", "hybrid_rerank"]
best = max(rag_names, key=lambda n: results[n]["macro_f1"])
print("Best RAG configuration by macro-F1:", best)
bp = preds_store[best]["pred"]
print(classification_report(gold_labels, bp, labels=["yes", "no", "maybe"], zero_division=0))
print(pd.DataFrame(confusion_matrix(gold_labels, bp, labels=["yes", "no", "maybe"]),
                   index=["true_yes", "true_no", "true_maybe"], columns=["pred_yes", "pred_no", "pred_maybe"]))

hit = np.array([eval_idx[j] in configs[best][j] for j in range(len(queries))])
correct = np.array([p == g for p, g in zip(bp, gold_labels)])
print(f"\nAccuracy when correct abstract retrieved in top-{TOP_K}: {correct[hit].mean():.3f} (n={hit.sum()})")
if (~hit).sum() > 0:
    print(f"Accuracy when it was NOT retrieved: {correct[~hit].mean():.3f} (n={(~hit).sum()})")

print("\nExample errors:")
shown = 0
for j in range(len(queries)):
    if not correct[j] and shown < 5:
        print(f"- Q: {queries[j]}\n  gold={gold_labels[j]} | pred={bp[j]} | gold abstract retrieved={bool(hit[j])}")
        shown += 1

In [ ]:
with open("results/predictions.json", "w") as f:
    json.dump({"queries": queries, "gold": gold_labels, "predictions": preds_store}, f)

import shutil
shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files
    files.download("results.zip")
except Exception as e:
    print("Download results.zip from the file panel on the left.", e)